# 05B CHARMM/CGenFF parameters (Route C)

Route C (CHARMM family): polymer **CGenFF** via CHARMM-GUI Ligand Reader & Modeler · protein **CHARMM36m** · water/ions **CHARMM TIP3P + SOD/CLA** · engine **GROMACS**.

Use Route C for protein–polymer systems that should stay in one consistent CHARMM force-field family. Use Route A (`05A_amber_gaff2_parameterisation.ipynb` → `06A_openmm_dry_polymer.ipynb` / `06B_openmm_solvated_system.ipynb`) for the Amber family (GAFF2 / ff19SB / OPC) in OpenMM.

**The enzyme–PHA production simulations (GK13/ANC45 × P3HO_4/P3HB_4) used this route.** CGenFF assigned all atom types, charges and parameters, so no GAFF2 charges enter them.

This notebook documents the manual CHARMM-GUI steps and provides two read-only checks. It does not automate CGenFF.

## CHARMM-GUI tools

- [CHARMM-GUI](https://www.charmm-gui.org)
- [Ligand Reader & Modeler](https://www.charmm-gui.org/?doc=input/ligandrm): CGenFF atom types, charges and parameters for a non-standard molecule.
- [Solution Builder](https://www.charmm-gui.org/?doc=input/solution): solvated box, ions and engine-specific inputs (GROMACS here).
- [Polymer Builder](https://www.charmm-gui.org/?doc=input/polymer): Choi et al., *J. Chem. Theory Comput.* 2021, [doi:10.1021/acs.jctc.1c00169](https://doi.org/10.1021/acs.jctc.1c00169). PHA monomer support not verified.

## Route as used

1. **Input:** an iPHASimulator R-configured SDF exported by `04_export_structures.ipynb` (e.g. `examples/output/polymer_structures/PHB4_R.sdf`).
2. **Ligand Reader & Modeler:** upload MOL/MOL2/SDF; residue name `LIG`; check chirality in the Marvin JS editor; keep hydrogens explicit.
3. **Outputs** (in `lig/`): `lig.rtf`, `lig_g.rtf`, `lig.prm`, `ndihe.str`, `lig.log`. The download also contains `toppar/` and a `gromacs/` folder (`topol.top`, `LIG.itp`, `charmm36.itp`).
4. **Solution Builder:** TODO(Zhiwen): box type and size, ion concentration, temperature.
5. **GROMACS** package → `06D_gromacs_solvated_system.ipynb` (planned).

## Open items

- TODO(Zhiwen): CGenFF version used for the production systems (read it from the `lig.rtf` / `lig.prm` header).
  *Evidence from the PHO4 Ligand Reader download (Apr 2026):* CGenFF program version 4.0 (released June 2024), for CGenFF topology and parameter files version 5.0.
- TODO(Zhiwen): which rtf (`lig.rtf` or `lig_g.rtf`) the GROMACS `LIG.itp` charges came from.
  *Evidence from the PHO4 Ligand Reader download (Apr 2026):* `lig.rtf` and `lig_g.rtf` give identical per-atom charges (99/99 atoms; `lig_g.rtf` only regroups them), and `gromacs/LIG.itp` matches both (net charge 0). Confirm for the production systems.
- TODO(Zhiwen): how the enzyme–polymer starting pose was obtained.

## Quality checklist

- **CGenFF penalties:** below 10 the analogy is fair; 10–50 basic validation is recommended; above 50 the parameters need extensive validation or optimisation (check A below).
- **Stereochemistry:** every backbone stereocentre must be R in the SDF sent to CHARMM-GUI (check B below) and still R after CHARMM-GUI (checked in 06D).
- **`[ defaults ]`** in the GROMACS topology is CHARMM-style: `1 2 yes 1.0 1.0`.
- **Net charge and ion counts** match the intended system (checked in 06D).

### Check A: CGenFF penalties

Set `LIG_DIR` to the `lig/` folder of a Ligand Reader & Modeler download. The check only reads `lig.rtf`, `lig_g.rtf` and `lig.prm`.

In [ ]:
from pathlib import Path
import sys

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "iphasimulator").exists())
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))
from iphasimulator.charmmgui_import import format_cgenff_penalty_report, read_cgenff_penalties

LIG_DIR = None  # e.g. Path("/path/to/PHO4_ligand_generator_charmm-gui-XXXXXXXXXX/lig")

if LIG_DIR is None:
    print("LIG_DIR is None: set it to a Ligand Reader & Modeler lig/ folder to check penalties.")
else:
    print(format_cgenff_penalty_report(read_cgenff_penalties(LIG_DIR)))

### Check B: stereocentres in the SDF sent to CHARMM-GUI

Set `SDF_PATH` to the SDF you uploaded. RDKit assigns CIP labels from the 3D coordinates; every backbone stereocentre should be R.

In [ ]:
from rdkit import Chem

SDF_PATH = None  # e.g. repo_root / "examples/output/polymer_structures/PHB4_R.sdf"

if SDF_PATH is None:
    print("SDF_PATH is None: set it to the SDF uploaded to Ligand Reader & Modeler.")
else:
    mol = Chem.MolFromMolFile(str(SDF_PATH), removeHs=False)
    Chem.AssignStereochemistryFrom3D(mol)
    centres = Chem.FindMolChiralCenters(mol, includeUnassigned=True, useLegacyImplementation=False)
    for index, label in centres:
        print(f"atom {index:>4} {mol.GetAtomWithIdx(index).GetSymbol()}: {label}")
    print(f"{len(centres)} stereocentres; all R: {bool(centres) and all(label == 'R' for _, label in centres)}")

## Files to keep

Keep the whole Ligand Reader & Modeler download (`lig/`, `toppar/`, `gromacs/`, `ligandrm.*`) together with the exact SDF you uploaded. Treat them as one matched set: atom names, charges and parameters in these files belong together.

## Next step

Build the solvated system in Solution Builder, download the GROMACS package, and validate it in `06D_gromacs_solvated_system.ipynb` (planned). For a quick vacuum check of the CGenFF polymer alone, use `06C_gromacs_dry_polymer.ipynb` (planned) with the Ligand Reader's `gromacs/` folder.

## Limitations

- CGenFF targets drug-like small molecules; PHA oligomer parameters are assigned by analogy.
- There is no PHA-specific validation of these parameters. Review penalties and, for production work, consider targeted validation of the highest-penalty terms.